# Kaggle Music Generator — Persistent ACE-Step Session
One Kaggle run watches public GitHub Issues and accepts multiple music requests. ACE-Step HTTP server uses --keep-loaded. Kaggle performs anonymous GET requests only; GitHub Actions handles all authenticated writes.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
print('Python:', sys.version, flush=True)
subprocess.run(['nvidia-smi'], check=True)
if shutil.which('nvcc') is None: raise RuntimeError('CUDA compiler nvcc is not available.')
print(subprocess.check_output(['nvcc','--version'], text=True).splitlines()[-1], flush=True)

In [ ]:
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','ninja-build','git','pkg-config','libcurl4-openssl-dev'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','hf','six'], check=True)
subprocess.run(['hf','--help'], check=True, stdout=subprocess.DEVNULL)

In [ ]:
engine_dir=Path('/kaggle/working/acestep.cpp')
if not engine_dir.exists(): subprocess.run(['git','clone','--depth','1','--recurse-submodules','--branch','master','https://github.com/ServeurpersoCom/acestep.cpp.git',str(engine_dir)], check=True)
else: subprocess.run(['git','submodule','update','--init','--recursive'], cwd=engine_dir, check=True)

In [ ]:
# Kaggle's CUDA runtime exposes CUDA libraries but may omit the driver CMake target required by GGML VMM. Disable VMM; CUDA kernels/cuBLAS remain enabled.
subprocess.run(['cmake','-S','.','-B','build','-DGGML_CUDA=ON','-DGGML_CUDA_NO_VMM=ON','-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_BUILD_TYPE=Release'], cwd=engine_dir, check=True)
subprocess.run(['cmake','--build','build','--config','Release','-j2'], cwd=engine_dir, check=True)
if not (engine_dir/'build'/'ace-server').is_file(): raise RuntimeError('Missing ace-server binary')

In [ ]:
# Q8_0 Turbo model set. ace-server loads models on first request and retains them with --keep-loaded.
subprocess.run(['bash','-lc','./models.sh --lm 4B --quant Q8_0'], cwd=engine_dir, check=True)
print('GGUF files:', [p.name for p in (engine_dir/'models').glob('*.gguf')], flush=True)

In [ ]:
repo_dir=Path('/kaggle/working/kaggle-music-generator')
if not repo_dir.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/unisocsu/kaggle-music-generator.git',str(repo_dir)], check=True)
subprocess.run([sys.executable,str(repo_dir/'kaggle'/'session_worker.py'),'--repo','unisocsu/kaggle-music-generator','--poll-seconds','15','--idle-seconds','180','--max-seconds',str(4*60*60+30*60)], cwd=engine_dir, check=True)
print('Session finished. Output files:', [p.name for p in Path('/kaggle/working/output').glob('*.mp3')], flush=True)